In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub  # ensure you have this installed: pip install kagglehub
from keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import random

# Download the dataset (if not already downloaded) and print the dataset path
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

# Set random seed for reproducibility
tf.random.set_seed(42)

# Define constants
IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32
EPOCHS = 15

# Define directories
train_dir = r"C:\Users\User\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Training"
test_dir = r"C:\Users\User\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Testing"

# Define class labels (must match subdirectory names)
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Create image data generator with augmentation and validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% data for validation
)

# Training generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=classes,  # ensure correct class ordering
    subset='training'
)

# Validation generator
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=classes,
    subset='validation'
)

print("Training samples:", train_generator.samples)
print("Validation samples:", validation_generator.samples)

# Build the CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(classes), activation='softmax')  # Output layer with softmax
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

# Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

# Plot training results
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Save the model
model.save('brain_tumor_model.h5')
print("Model saved as 'brain_tumor_model.h5'")

# Load model for evaluation
model = load_model('brain_tumor_model.h5')

# ==============================
# TESTING & EVALUATION
# ==============================

# Test Data Generator (no augmentation, only rescaling)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Important to maintain label order
)

# Evaluate model on test data
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# Predict on test set
predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Classification Report
print("\nClassification Report:\n")
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

# Confusion Matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

# ==============================
# DISPLAY SAMPLE PREDICTIONS
# ==============================

# Get random test images for visualization
filenames = test_generator.filenames
random_indices = random.sample(range(len(filenames)), 12)

plt.figure(figsize=(12, 12))
for i, idx in enumerate(random_indices):
    img_path = os.path.join(test_dir, filenames[idx])
    img = plt.imread(img_path)

    predicted_label = class_labels[predicted_classes[idx]]
    true_label = class_labels[true_classes[idx]]

    plt.subplot(4, 3, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Pred: {predicted_label}\nTrue: {true_label}", 
              color=("green" if predicted_label == true_label else "red"))

plt.tight_layout()
plt.show()
# ==============================
# CLASSIFY NEW IMAGES FROM A DIRECTORY
# ==============================

from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
import os

# Path to the new images for classification
new_images_dir = r"C:\pythonProj\Brain\tumor\classify"

# Function to preprocess and predict image
def predict_image(img_path, model, img_size=(IMG_HEIGHT, IMG_WIDTH)):
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img) / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    
    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction)  # Get class index
    confidence = np.max(prediction) * 100  # Get confidence percentage
    
    return predicted_class, confidence, img

# Get list of images
image_files = [f for f in os.listdir(new_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Set up the plot
plt.figure(figsize=(12, 8))
rows = (len(image_files) // 4) + 1  # Adjust rows dynamically

for i, img_file in enumerate(image_files):
    img_path = os.path.join(new_images_dir, img_file)
    predicted_class, confidence, img = predict_image(img_path, model)
    
    # Display image with predicted class
    plt.subplot(rows, 4, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{classes[predicted_class]}\n({confidence:.2f}%)")

plt.tight_layout()
plt.show()


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1
Found 4571 images belonging to 4 classes.
Found 1141 images belonging to 4 classes.
Training samples: 4571
Validation samples: 1141
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 74, 74, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
     